# 01 — Exploratory Data Analysis

Mục tiêu: kiểm tra cấu trúc dữ liệu, giao dịch âm/return, phân phối SKU và profit weight. Notebook này **không train model**.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import *
from src.pipeline import load_train_clean, aggregate_daily_y, compute_sku_weights

In [ ]:
df = load_train_clean(TRAIN_PATH)
print(df.shape)
display(df.head())
print(df['Date'].min(), '→', df['Date'].max())
print('n_sku:', df['ItemCode'].nunique())

## Dataset overview

In [ ]:
overview = pd.DataFrame({
    'metric': ['rows', 'n_skus', 'date_min', 'date_max', 'quantity_negative_rows', 'quantity_zero_rows', 'quantity_positive_rows'],
    'value': [
        len(df),
        df['ItemCode'].nunique(),
        df['Date'].min(),
        df['Date'].max(),
        int((df['Quantity'] < 0).sum()),
        int((df['Quantity'] == 0).sum()),
        int((df['Quantity'] > 0).sum()),
    ]
})
display(overview)

## Negative Quantity / return-like transactions

In [ ]:
neg = df[df['Quantity'] < 0].copy()
print('Negative rows:', len(neg))
display(neg[['Quantity', 'SalesAmount', 'Cost Amount', 'line_profit']].describe().T)
display(neg.groupby('ItemCode')['Quantity'].agg(['count', 'sum']).sort_values('count', ascending=False).head(20))

## Long-tail SKU distribution

In [ ]:
daily = aggregate_daily_y(df)
sku_sales = daily.groupby('ItemCode')['y'].agg(['sum', 'mean', 'count']).reset_index()
sku_sales = sku_sales.sort_values('sum', ascending=False)
display(sku_sales.head(20))

plt.figure(figsize=(10, 4))
plt.plot(np.arange(1, len(sku_sales)+1), sku_sales['sum'].cumsum() / max(sku_sales['sum'].sum(), 1))
plt.xlabel('SKU rank by demand')
plt.ylabel('Cumulative demand share')
plt.title('Long-tail demand distribution')
plt.grid(True, alpha=0.3)
plt.show()

## Profit weight concentration

In [ ]:
weight_df = compute_sku_weights(df).sort_values('weight', ascending=False).reset_index(drop=True)
weight_df['cum_weight'] = weight_df['weight'].cumsum()
display(weight_df.head(20))

thresholds = [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]
rows = []
for t in thresholds:
    rows.append({'cum_weight_threshold': t, 'n_skus': int((weight_df['cum_weight'] < t).sum() + 1)})
display(pd.DataFrame(rows))

## Daily demand trend

In [ ]:
daily_total = daily.groupby('Date', as_index=False)['y'].sum()
plt.figure(figsize=(12,4))
plt.plot(daily_total['Date'], daily_total['y'])
plt.title('Total daily positive demand')
plt.xlabel('Date')
plt.ylabel('Quantity')
plt.grid(True, alpha=0.3)
plt.show()

daily_total['dow'] = daily_total['Date'].dt.day_name()
display(daily_total.groupby('dow')['y'].agg(['mean', 'median', 'sum']).sort_values('mean', ascending=False))